In [1]:
import sleap
import numpy as np
import matplotlib.pyplot as plt
import os, sys
import datetime

sys.path.append('..')
sys.path.append('/home/mingxiao/Desktop/jelly-sleap/sleap/jelly/')

from python.postprocess import *
from python.animation import *
from autoencoder.src.data_loader import *
from autoencoder.min2.run_models import *

from tensorflow.keras.datasets import fashion_mnist

In [ ]:
# TODO: use an artificial dataset 
# TODO: fourier transform/PCA; reconstruct weights of the PCA 
# plot the heatmap of each time window (batch by time); roaster
# downsample in time in the VAE; or just flatten the time dimension into feature dimension

# Generate artificial jellyfish dataset

canvas size = (200, 200)

center position = (100, 100)

tentacle bulb count = 17

max radius = 60

period for one swim pulse = 20 - 50 frames 

P(swim) = 0.9, P(still) = 0.1 

period of still = 5 - 20 frames

contraction ratio = 0.4 - 0.8

frame count = 10000

Assume no rotation, i.e., each point is always on the same radius. At each time stamp, radius of all points are the same. 

In [16]:
tb_cnt = 3
frame_cnt = 5
radius = np.arange(1, 6)
radius 
# array([1, 2, 3, 4, 5]) -> need 

array([1, 2, 3, 4, 5])

In [21]:
np.broadcast_to(radius[:, None], (frame_cnt, tb_cnt))

array([[1, 1, 1],
       [2, 2, 2],
       [3, 3, 3],
       [4, 4, 4],
       [5, 5, 5]])

In [24]:
def generate_jellyfish_dataset(
    tb_cnt=17, 
    frame_cnt=10000, 
    max_r=60, 
    p_swim=0.9, 
    t_swim=(20, 50), 
    t_still=(5, 20), 
    contract_ratio=(0.4, 0.8), 
    center_pos=(100, 100)):
    
    radius_data = np.zeros(frame_cnt)
    radius_data[0] = max_r # start from max radius 
    frame_idx = 1
    while frame_idx < frame_cnt:
        # Generate swim or still state
        if np.random.rand() < p_swim: # Swim state
            swim_duration = np.random.randint(*t_swim)
            swim_duration = swim_duration - 1 if swim_duration % 2 == 1 else swim_duration # make sure the duration is even 
            swim_start = frame_idx
            swim_end = min(frame_idx + swim_duration, frame_cnt)
            swim_mid = min(swim_start + swim_duration // 2, frame_cnt)
            min_r = np.random.uniform(*contract_ratio)
            r_series = np.linspace(max_r, min_r, swim_duration // 2)
            radius_data[swim_start : swim_mid] = r_series[:swim_mid - swim_start]
            if swim_mid >= frame_cnt:
                break
            radius_data[swim_mid : swim_end] = r_series[::-1][:swim_end - swim_mid]
            frame_idx = swim_end
        else: # still state
            still_duration = np.random.randint(*t_still)
            still_end = min(frame_idx + still_duration, frame_cnt)
            radius_data[frame_idx : still_end] = radius_data[frame_idx - 1]
            frame_idx = still_end
    
    radius_data = np.broadcast_to(radius_data[:, None], (frame_cnt, tb_cnt))
    theta_data = np.linspace(-np.pi, np.pi, tb_cnt, endpoint=False)
    
    dataset = np.zeros((frame_cnt, tb_cnt, 2))
    dataset[:, :, 0] = radius_data * np.cos(theta_data)
    dataset[:, :, 1] = radius_data * np.sin(theta_data)
    dataset += center_pos
    
    return dataset            

In [25]:
toy_jelly = generate_jellyfish_dataset()
print(toy_jelly.shape)

(10000, 17, 2)


In [ ]:
toy_jelly_animation_path = '/home/mingxiao/Desktop/animation/toy_jelly_v0.mp4'
get_animation_from_tracked_points(toy_jelly, 200, 200, 50, bg_video=None, bg_video_start_idx=0, output_path=toy_jelly_animation_path);

INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
INFO:matplotlib.animation:MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 800x800 -pix_fmt rgba -r 50 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/mingxiao/Desktop/animation/toy_jelly_v0.mp4


In [4]:
(x_train, _), (x_test, _) = fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print (x_train.shape)
print (x_test.shape)

4431872/4422102 [==============================] - 0s 0us/step
(60000, 28, 28)
(10000, 28, 28)


In [11]:
def get_ae_model_1(input_shape=(28, 28), latent_dim=32, num_layers=4):
    # simple feedforward autoencoder
    inputs = Input(shape=input_shape)
    layer_dim = latent_dim * (2 ** num_layers)
    
    x = Flatten()(inputs)
    # encoder
    for _ in range(num_layers):
        x = Dense(layer_dim, activation='relu')(x)
        layer_dim //= 2
    x = Dense(latent_dim, activation='relu')(x)
    
    # decoder
    for _ in range(num_layers):
        x = Dense(layer_dim, activation='relu')(x)
        layer_dim *= 2
    
    outputs = Dense(input_shape[0] * input_shape[1], activation='sigmoid')(x)
    outputs = Reshape(input_shape)(outputs)

    model = Model(inputs, outputs)
    model.summary()
    
    return model

In [12]:
m1 = get_ae_model_1(latent_dim=64, num_layers=2)
m1.compile(optimizer='adam', loss='mse')

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 28, 28)]          0         
                                                                 
 flatten_2 (Flatten)         (None, 784)               0         
                                                                 
 dense_18 (Dense)            (None, 256)               200960    
                                                                 
 dense_19 (Dense)            (None, 128)               32896     
                                                                 
 dense_20 (Dense)            (None, 64)                8256      
                                                                 
 dense_21 (Dense)            (None, 64)                4160      
                                                                 
 dense_22 (Dense)            (None, 128)               8320

In [14]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1)

In [15]:
m1.fit(x_train, x_train,
                epochs=100,
                shuffle=True,
                validation_data=(x_test, x_test), 
                callbacks=[tensorboard_callback, early_stopping_callback])

Epoch 1/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0105 - val_loss: 0.0104
Epoch 2/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0102 - val_loss: 0.0101
Epoch 3/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0100 - val_loss: 0.0100
Epoch 4/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0097 - val_loss: 0.0099
Epoch 5/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0095 - val_loss: 0.0096
Epoch 6/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0094 - val_loss: 0.0095
Epoch 7/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0092 - val_loss: 0.0093
Epoch 8/100
1875/1875 [==============================] - 9s 5ms/step - loss: 0.0091 - val_loss: 0.0093
Epoch 9/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0090 - val_loss: 0.0094
Epoch 10/100
1875/1875 [==============================] - 9s 5ms/